# 7. Learn and evaluate a baseline

**Open:** restart R and save the supplied script as `scripts/my-07-baseline.R`.

**Input:** `outputs/checkpoints/farm-split.rds` from Chapter 6. **Result:** a score table, predictions and a plot you can explain.

## 7.1 Fit one transparent rule

Run the supplied `load-split` block to restore the training and test tables:

In [1]:
if (!file.exists("outputs/checkpoints/farm-split.rds")) stop("Run Chapter 6 first to create farm-split.rds.")
split_checkpoint <- readRDS("outputs/checkpoints/farm-split.rds")
train <- split_checkpoint$train
test <- split_checkpoint$test
farm_ids <- split_checkpoint$farm_ids
test_farms <- split_checkpoint$test_farms

Compute a mean using **training outcomes only**, then repeat it for each test record:

In [2]:
baseline_log10 <- mean(train$log10_msc)
baseline_log10
test$prediction_log10 <- rep(baseline_log10, nrow(test))
stopifnot(all(test$prediction_log10 == mean(train$log10_msc)))

[1] 0.7450425

The learned value is about **0.7450 log10 CFU/mL**. Every held-out sample receives it. No weather or management predictors enter this rule.

## 7.2 Read the result table

A residual is observed minus predicted. As in the hand example, MAE averages absolute residuals; RMSE averages their squares and takes a square root.

In [3]:
test$error_log10 <- test$log10_msc - test$prediction_log10
rmse <- sqrt(mean(test$error_log10^2))
mae <- mean(abs(test$error_log10))
data.frame(model = "Training-mean baseline", n_test = nrow(test),
           RMSE_log10_CFU_ml = rmse, MAE_log10_CFU_ml = mae)

model,n_test,RMSE_log10_CFU_ml,MAE_log10_CFU_ml
<chr>,<int>,<dbl>,<dbl>
Training-mean baseline,105,0.5289231,0.409148


**How to read these numbers:**

- **105:** the number of held-out test records, from 21 farms. Each record contributes one equally weighted error.
- **RMSE about 0.529; MAE about 0.409:** prediction errors on the **log10** target scale.
- **Their role:** a starting point for evaluating later procedures on an appropriate validation design.

RMSE 0.529 is not a 52.9% error rate. An error of 1 on a log10 scale corresponds to a tenfold concentration ratio, but RMSE should not be relabeled as a typical fold error without further definition. The [two-row hand calculation](04-small-farm-example.ipynb) explains the operations on an easier scale.

## 7.3 Reconstruct one error yourself

Run the supplied `independent` check. It also displays three observed values, predictions and residuals:

In [4]:
rmse_check <- sqrt(sum((test$log10_msc - baseline_log10)^2) / nrow(test))
mae_check <- sum(abs(test$log10_msc - baseline_log10)) / nrow(test)
stopifnot(isTRUE(all.equal(rmse, rmse_check)))
stopifnot(isTRUE(all.equal(mae, mae_check)))
round(data.frame(observed = head(test$log10_msc, 3),
                 predicted = head(test$prediction_log10, 3),
                 residual = head(test$error_log10, 3)), 4)

observed,predicted,residual
<dbl>,<dbl>,<dbl>
0.8129,0.745,0.0679
0.5441,0.745,-0.2010
-0.3010,0.745,-1.0461


Choose the first row. Subtract predicted from observed by hand and compare with its residual. Display rounding can cause a small difference; the automated check uses full precision.

**Ask AI:** give it the actual columns and your hand calculation. Ask it to reconstruct RMSE without refitting the model or changing the split. Compare its answer with your saved observations and predictions.

## 7.4 Save and inspect the evidence

Select **all remaining supplied blocks**, from `sensitivity` through `audit-save`, and Run. They perform an additional target-limit check and save the outputs. The blocks still run even if you leave their technical details collapsed.

**Additional check — `sensitivity`:** the same fixed predictions are evaluated on the quantified test subset. This describes another subset; it does not recover true below-detection concentrations or fix that limitation.

In [5]:
# Evaluation subset only: the baseline is not refitted or chosen using this result.
quantified_test <- test[!test$left_censored, ]
quantified_rmse <- sqrt(mean(quantified_test$error_log10^2))
data.frame(evaluation = c("All eligible test records", "Quantified test records only"),
           n = c(nrow(test), nrow(quantified_test)),
           RMSE_log10_CFU_ml = c(rmse, quantified_rmse))

evaluation,n,RMSE_log10_CFU_ml
<chr>,<int>,<dbl>
All eligible test records,105,0.5289231
Quantified test records only,104,0.5062914


**File export — `save`:**

In [6]:
dir.create("outputs/baseline", recursive = TRUE, showWarnings = FALSE)
score <- data.frame(model = "Training-mean baseline", training_mean_log10 = baseline_log10,
                    train_rows = nrow(train), test_rows = nrow(test),
                    train_farms = length(unique(train$FarmID)), test_farms = length(test_farms),
                    RMSE_log10_CFU_ml = rmse, MAE_log10_CFU_ml = mae,
                    quantified_test_n = nrow(quantified_test), quantified_test_RMSE = quantified_rmse)
write.csv(score, "outputs/baseline/scores.csv", row.names = FALSE)
split_table <- data.frame(FarmID = farm_ids,
                         split = ifelse(farm_ids %in% test_farms, "test", "train"))
write.csv(split_table, "outputs/baseline/farm-split.csv", row.names = FALSE)
write.csv(test[c("FarmID", "Sampling", "left_censored", "log10_msc", "prediction_log10", "error_log10")],
          "outputs/baseline/test-predictions.csv", row.names = FALSE)
writeLines(capture.output(sessionInfo()), "outputs/baseline/session-info.txt")
file.exists("outputs/baseline/scores.csv")

[1] TRUE

**Plot export — `plot`:**

In [7]:
png("outputs/baseline/training-distribution.png", width = 1200, height = 720, res = 150)
par(mar = c(4.5, 4.5, 2.5, 1), family = "sans", col.axis = "#294943", col.lab = "#294943")
hist(train$log10_msc, breaks = 18, col = "#8cc7b6", border = "white",
     main = "The baseline is learned from training farms",
     xlab = "Training log10 MSC (CFU/mL)", ylab = "Number of records")
abline(v = baseline_log10, col = "#a94c31", lwd = 3)
legend("topright", "Training mean", col = "#a94c31", lwd = 3, bty = "n")
invisible(dev.off())
cat("Saved outputs/baseline/training-distribution.png\n")

Saved outputs/baseline/training-distribution.png


![Training log10 MSC distribution with the training mean marked by a vertical line.](../outputs/baseline/training-distribution.png)

The plot is written directly to `outputs/baseline/training-distribution.png`. Open that file in Files; it may not appear in the Plots pane.

**Selection record — `audit-save`:**

In [8]:
write.csv(split_checkpoint$audit, "outputs/baseline/data-audit.csv", row.names = FALSE)
cat("All data and baseline checks passed.\n")

All data and baseline checks passed.


| Find in outputs/baseline | What you can check |
|---|---|
| `scores.csv` | The fitted mean and the two error metrics |
| `farm-split.csv` | Which role each farm was assigned |
| `test-predictions.csv` | Observed values, predictions and residuals |
| `training-distribution.png` | Where the learned mean lies in the training distribution |

## 7.5 Write a conclusion that fits the evidence

Write two sentences: name the rule and its test error, then state one important limit. Keep the target scale and held-out setting explicit.

```{admonition} Reference write-up — open after you try
:class: tip dropdown
A training-mean baseline achieved RMSE about 0.529 and MAE about 0.409 log10 CFU/mL on 105 records from 21 held-out farms. This is a fixed-split microbial-quality benchmark; below-detection target substitutions and the represented study setting limit its interpretation. It does not establish a causal effect or a milk-safety decision.
```

Before making a stronger claim, ask what comparison, acceptable error or study design it would require. A single baseline score does not establish an application-ready predictor.

**Finish:** locate your score, plot and AI-use record, explain one residual, then restart R and run a my- copy of `scripts/worked-example.R` from the beginning. It should recreate the real-data outputs.